# RDD2022 Data Loading and Preprocessing

This notebook prepares the RDD2022 dataset for YOLOv8 training:
- EDA: class distribution and image sizes
- Label cleanup and validation
- Pascal VOC XML to YOLO TXT conversion
- Train/val/test split (80/10/10) from labeled data
- Visual verification of annotations
- Create data.yaml

## 1) Configuration

In [ ]:
import os
from pathlib import Path
import random
import shutil
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Paths in Kaggle
DATA_ROOT = Path("/kaggle/input/rdd2022")
WORK_ROOT = Path("/kaggle/working")
OUT_ROOT = WORK_ROOT / "rdd2022_yolo"

# Expected dataset splits (auto-detected if different)
EXPECTED_SPLITS = ["train", "val", "test"]
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

OUT_ROOT

## 2) Collect image and annotation paths

In [ ]:
def find_split_dirs(root: Path):
    return {d.name: d for d in root.iterdir() if d.is_dir()}

def get_images_labels(split_dir: Path):
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"
    if not lbl_dir.exists():
        lbl_dir = split_dir / "annotations" / "xmls"
    return img_dir, lbl_dir

def gather_pairs_by_split(data_root: Path):
    split_dirs = find_split_dirs(data_root)
    pairs_by_split = {}
    for split_name, split_dir in split_dirs.items():
        img_dir, lbl_dir = get_images_labels(split_dir)
        if not img_dir.exists() or not lbl_dir.exists():
            continue
        label_files = sorted(list(lbl_dir.glob("*.xml")) + list(lbl_dir.glob("*.txt")))
        pairs = []
        for label_path in label_files:
            img_path = img_dir / (label_path.stem + ".jpg")
            if img_path.exists():
                pairs.append((img_path, label_path, split_name))
        pairs_by_split[split_name] = pairs
    return pairs_by_split

pairs_by_split = gather_pairs_by_split(DATA_ROOT)
pairs = [p for split in pairs_by_split.values() for p in split]
pairs_by_split.keys(), len(pairs)

## 3) EDA: class distribution and image sizes

In [ ]:
def parse_xml(xml_path: Path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    w = int(size.findtext("width"))
    h = int(size.findtext("height"))
    objects = []
    for obj in root.findall("object"):
        name = obj.findtext("name")
        bbox = obj.find("bndbox")
        xmin = int(float(bbox.findtext("xmin")))
        ymin = int(float(bbox.findtext("ymin")))
        xmax = int(float(bbox.findtext("xmax")))
        ymax = int(float(bbox.findtext("ymax")))
        objects.append((name, xmin, ymin, xmax, ymax))
    return (w, h), objects

def parse_yolo_txt(label_path: Path):
    objects = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id = int(float(parts[0]))
            objects.append((f"CLASS_{cls_id}",))
    return objects

class_counts = Counter()
sizes = []

for img_path, label_path, _ in pairs:
    if label_path.suffix.lower() == ".xml":
        (w, h), objects = parse_xml(label_path)
    else:
        with Image.open(img_path) as img:
            w, h = img.size
        objects = parse_yolo_txt(label_path)
    sizes.append((w, h))
    for name, *_ in objects:
        class_counts[name] += 1

class_counts.most_common(20)

In [ ]:
# Plot class distribution
labels, counts = zip(*class_counts.most_common())
plt.figure(figsize=(10, 4))
plt.bar(labels, counts)
plt.title("Class Distribution")
plt.xticks(rotation=45)
plt.show()

# Image size stats
widths = [s[0] for s in sizes]
heights = [s[1] for s in sizes]
print({"width_min": min(widths), "width_max": max(widths), "height_min": min(heights), "height_max": max(heights)})

## 4) Label cleanup and validation

In [ ]:
def normalize_class(name: str) -> str:
    return name.strip().replace(" ", "_").upper()

# Build class list dynamically from data
class_set = {normalize_class(name) for name in class_counts.keys()}
classes = sorted(class_set)
class_to_id = {c: i for i, c in enumerate(classes)}
classes

## 5) Use existing splits (fallback to 80/10/10 if missing)

In [ ]:
if all(k in pairs_by_split for k in EXPECTED_SPLITS):
    train_pairs = pairs_by_split["train"]
    val_pairs = pairs_by_split["val"]
    test_pairs = pairs_by_split["test"]
else:
    # Fallback: create split from all labeled pairs
    random.shuffle(pairs)
    n_total = len(pairs)
    n_train = int(n_total * 0.80)
    n_val = int(n_total * 0.10)
    train_pairs = pairs[:n_train]
    val_pairs = pairs[n_train:n_train + n_val]
    test_pairs = pairs[n_train + n_val:]

len(train_pairs), len(val_pairs), len(test_pairs)

## 6) Convert XML to YOLO TXT

In [ ]:
def clamp(v, vmin, vmax):
    return max(vmin, min(vmax, v))

def convert_bbox_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
    xmin = clamp(xmin, 0, img_w - 1)
    ymin = clamp(ymin, 0, img_h - 1)
    xmax = clamp(xmax, 0, img_w - 1)
    ymax = clamp(ymax, 0, img_h - 1)
    if xmax <= xmin or ymax <= ymin:
        return None
    x_center = (xmin + xmax) / 2.0 / img_w
    y_center = (ymin + ymax) / 2.0 / img_h
    w = (xmax - xmin) / img_w
    h = (ymax - ymin) / img_h
    return x_center, y_center, w, h

def write_yolo_labels(pairs_list, split_name):
    img_out_dir = OUT_ROOT / "images" / split_name
    lbl_out_dir = OUT_ROOT / "labels" / split_name
    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    stats = defaultdict(int)
    for img_path, label_path, _ in pairs_list:
        if label_path.suffix.lower() == ".txt":
            shutil.copy2(img_path, img_out_dir / img_path.name)
            shutil.copy2(label_path, lbl_out_dir / f"{img_path.stem}.txt")
            stats["copied_txt"] += 1
            continue
        (w, h), objects = parse_xml(label_path)
        yolo_lines = []
        for name, xmin, ymin, xmax, ymax in objects:
            norm_name = normalize_class(name)
            if norm_name not in class_to_id:
                stats["skipped_class"] += 1
                continue
            bbox = convert_bbox_to_yolo(xmin, ymin, xmax, ymax, w, h)
            if bbox is None:
                stats["invalid_bbox"] += 1
                continue
            cls_id = class_to_id[norm_name]
            x, y, bw, bh = bbox
            yolo_lines.append(f"{cls_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}")
        if not yolo_lines:
            stats["empty_labels"] += 1
            continue
        # Copy image and write label file
        shutil.copy2(img_path, img_out_dir / img_path.name)
        with open(lbl_out_dir / f"{img_path.stem}.txt", "w") as f:
            f.write("\n".join(yolo_lines))
        stats["written_xml"] += 1
    return stats

# Clean output dir
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

train_stats = write_yolo_labels(train_pairs, "train")
val_stats = write_yolo_labels(val_pairs, "val")
test_stats = write_yolo_labels(test_pairs, "test")
train_stats, val_stats, test_stats

## 7) Create data.yaml

In [ ]:
data_yaml = OUT_ROOT / "data.yaml"
data_yaml.write_text("\n".join([
    f"path: {OUT_ROOT}",
    "train: images/train",
    "val: images/val",
    "test: images/test",
    f"nc: {len(classes)}",
    "names:",
    *[f"  - {c}" for c in classes],
 ]))
data_yaml

## 9) Notes
- If your dataset already contains YOLO TXT labels, they are copied directly without conversion.
- If labels are XML (Pascal VOC), they are converted to YOLO format.
- The original dataset test split (if unlabeled) can be used for final inference only.